In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.linear_model import LinearRegression
import shap
from time import perf_counter

/Users/maxi/miniconda3/envs/EML/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load arbitrary data

In [2]:
data = pd.read_parquet("../data/california_housing_prices.parquet")
X = data.drop(columns="HousePrice")
y = data["HousePrice"]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Fit ML model

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

# Select background data and data sample(s)

In [5]:
n_background = 100
n_samples = 3

X_background = X_train.sample(n=n_background, random_state=42) # if n too large shap library will alter background data so it wont equal our calculated one
x = X_test.iloc[0:n_samples]

# Configure and compute Shapley Values

In [6]:
%run ../shap_implementation/exact_explainer.py

## Shapley Values

Formally, Shapley values are defined as

$$
\phi_i(v) = \sum_{S \subseteq N\setminus \{i\}} 
\frac{|S|! (n - |S| - 1)!}{n!}
\left[v(S \cup \{i\}) - v(S)\right]
$$

where $N$ is the set of all features, e.g. $N = \{\text{MedIncome}, \text{AveRooms}, \text{Population}, \ldots\}$, $n = |N|$ and $S$ is a subset of features that does not contain the target feature $i$. The coalition function $v(S)$ calculates the model prediction given only the features contained in the subset $S$. Finally, $\phi_i(v)$ represents the Shapley value, i.e. the marginal contribution of feature $i$ to the model prediction.

Hence, the marginal contribution of a feature $i$ is calculated as the weighted sum of all possible coalition differences between subsets containing $i$ and the corresponding subsets not containing $i$. The difference

$$
v(S \cup \{i\}) - v(S)
$$

therefore measures how much the model prediction changes when feature $i$ is added to the coalition $S$.

Note that

$$
\sum_{S \subseteq N\setminus \{i\}}
\frac{|S|! (n - |S| - 1)!}{n!}
$$

always sums to $1$. Consequently, the Shapley value can be interpreted as a weighted average of the marginal contribution of feature $i$ across all possible coalitions.

Essentially, the marginal contribution of a feature is determined by comparing the model prediction when the feature is absent from a coalition with the prediction when the feature is added to that coalition. This ensures that the contribution of a feature is evaluated across all possible combinations of the remaining features rather than for only a single coalition.


In [7]:
exact_explainer = ExactExplainer(model, X_background)
exact_shap_values = exact_explainer.explain(x)

In [8]:
for i, shap_value in enumerate(exact_shap_values):
    print(f"SHAP values for row {i}: {shap_value} \n")

SHAP values for row 0: [ 57.95393647  13.18218987   3.71649552  -7.88759643  -0.25701394
   0.17573852  67.00902979 -50.15454822] 

SHAP values for row 1: [ 46.16660829   6.17452839   0.35256677 -12.19112668  -0.32585782
   0.20493624 -59.50195173  89.73606114] 

SHAP values for row 2: [ 4.94123943e+01 -7.84079456e+00 -1.38079739e+01 -6.29635282e+00
 -1.22696163e-01 -2.53363013e-02  3.39566112e+01  2.91035874e+01] 



## Compute and verify property Efficiency for sample $x_0$

We compute the expected prediction over the background dataset with size |M|,
$$
E[f(X)] = \frac{1}{|M|}\sum_{x' \in M} f(x'),
$$
and add the sum of all SHAP values. The efficiency property states that this must reconstruct the model prediction for the observation $x$:
$$
E[f(X)] + \sum_{i=1}^{n}\phi_i(v) = f(x).
$$
Thus, the expected prediction represents the base value, while the SHAP values represent the individual feature contributions that collectively account for the difference between the base value and the actual prediction.

But why does this hold?

Equivalently,

$$
\sum_{i=1}^{n}\phi_i(v) = f(x) - E[f(X)] 
$$

Substituting the original shapley formula

$$
\sum_{i=1}^{n} \sum_{S \subseteq N\setminus \{i\}}  \frac{|S|! (n - |S| - 1)!}{n!} \left[v(S \cup \{i\}) - v(S)\right] = f(x) - E[f(X)] 
$$

For any ordering $(i_1, i_2, \dots, i_n)$, the corresponding marginal contributions are
$$
[v({i_1})-v(\emptyset)] + [v({i_1,i_2})-v({i_1})] + \cdots + [v(N)-v(N\setminus{i_n})].
$$

It becomes obvious that each intermediate coalition appears once positively and once negatively: they cancel each other out.

Therefore,
$$
\sum_{i=1}^{n} \sum_{S \subseteq N\setminus \{i\}}  \frac{|S|! (n - |S| - 1)!}{n!} \left[v(S \cup \{i\}) - v(S)\right] = v(N) - v(\emptyset)
$$

For the interventional coalition function $E[f(X) | \text{ do } X_s = x_s]$, we get

$$
v(\emptyset) = E[f(X) | \text{ do } \emptyset] = E[f(X)],

v(N) = E[f(X) | \text{ do } X_s = x_s] = f(x) 
$$

since for the empty set $\emptyset$ there is no intervention and we dont change any feature values in the background data distribution. As we dont change anything, the outcome is simply the expected model prediction over the background data, essentially the mean model prediction over the backgriund distribution. 

For the case where we evaluate all features $S \cup \{i\} = N$, we set every feature value in the background distribution to the observed value from our sample $x$. Since every background value for every feature is now the same, the mean model prediction over the background data is now equal to the model prediction for our sample $x$: $f(x)$.

Consequently, 
$$
\sum_{i=1}^{n} \sum_{S \subseteq N\setminus \{i\}}  \frac{|S|! (n - |S| - 1)!}{n!} \left[v(S \cup \{i\}) - v(S)\right] = \sum_{i=1}^{n}\phi_i(v) = f(x) - E[f(X)]
$$

In [9]:
expected_value = exact_explainer.expected_value
f_x = model.predict(x.iloc[[0]])

print(
    f"{'Expected value E[f(x)]':<45}: {expected_value:.6f}\n"
    f"{'SHAP values for x_0':<45}: {exact_shap_values[0]}\n"
    f"{'Sum of SHAP values for x_0':<45}: {sum(exact_shap_values[0]):.6f}\n"
    f"{'Model prediction f(x) for x_0':<45}: {f_x[0]:.6f}\n"
    f"{'Expected value + summed SHAP values':<45}: "
    f"{expected_value + sum(exact_shap_values[0]):.6f}"
)

Expected value E[f(x)]                       : 193.917715
SHAP values for x_0                          : [ 57.95393647  13.18218987   3.71649552  -7.88759643  -0.25701394
   0.17573852  67.00902979 -50.15454822]
Sum of SHAP values for x_0                   : 83.738232
Model prediction f(x) for x_0                : 277.655947
Expected value + summed SHAP values          : 277.655947


## Test value computation efficiency

In [10]:
print("Runtime")
print("-" * 45)

start_t = perf_counter()
exact_explainer = ExactExplainer(model, X_background)
exact_shap_values = exact_explainer.explain(x)
print(f"{'Simple Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

start_t = perf_counter()
optimized_exact_explainer = ExactExplainerOptimized(model, X_background)
optimized_exact_shap_values = optimized_exact_explainer.explain(x)
print(f"{'Optimized Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

start_t = perf_counter()
shap_explainer = shap.ExactExplainer(model.predict, X_background)
shap_values = shap_explainer(x)
print(f"{'SHAP Library':<30}: {perf_counter() - start_t:.3f}s")


results = []

for i, (pred_a, pred_b, pred_c) in enumerate(zip(exact_shap_values, optimized_exact_shap_values, shap_values.values)):
    results.append({
        "Sample": f"x_{i}",
        "Simple Exact": sum(pred_a),
        "Optimized Exact": sum(pred_b),
        "SHAP Library": sum(pred_c),
    })

print("\n \nSHAP Summed Value Comparison")
print("-" * 60)

pd.DataFrame(results).set_index("Sample").round(6)

Runtime
---------------------------------------------
Simple Exact Explainer        : 5.088s
Optimized Exact Explainer     : 0.524s
SHAP Library                  : 3.722s

 
SHAP Summed Value Comparison
------------------------------------------------------------


,Simple Exact,Optimized Exact,SHAP Library
Sample,,,
x_0,83.738232,83.738232,83.738232
x_1,70.615765,70.615765,70.615765
x_2,84.379439,84.379439,84.379439


# Limits of Exact Explainer

The computational cost of the Exact Explainer depends on both the number of samples to be explained and the size of the background dataset. Let $\mathrm{h}$ denote the number of samples to explain, $\mathrm{M}$ the number of background samples and $\mathrm{n}$ the number of features.

For a fixed number of features and background samples, the computational cost scales approximately linearly with the number of samples to explain:

$$
T(h) = O(h).
$$

Thus, if $\mathrm{h}$ samples require $\mathrm{x}$ seconds, then $\mathrm{10h}$ samples will require approximately $\mathrm{(10h/h)x}$ seconds, assuming all other factors remain constant.

The background dataset affects the cost of each coalition evaluation because the model prediction is computed over all $\mathrm{M}$ background samples. In principle, this introduces an approximately linear dependence on $\mathrm{M}$:

$$
T(M) = O(M).
$$

In practice, however, this dependence may be less noticeable for moderate background sizes. Modern machine-learning libraries perform predictions in batches using vectorized and highly optimized implementations, so increasing $\mathrm{M}$ does not necessarily result in a proportional increase in wall-clock time for small or moderate values of $\mathrm{M}$. The SHAP library also commonly uses a bounded background sample size of $100$, which limits this source of computational growth.

The dominant limitation of the Exact Explainer is instead the exponential dependence on the number of features. For a feature $\mathrm{i}$, its exact Shapley value requires evaluating the marginal contribution of $\mathrm{i}$ for every subset $S \subseteq N \setminus \{i\}$. With $\mathrm{n}$ features, there are

$$
2^{n-1}
$$

such subsets for each feature. Consequently, computing the exact Shapley values for all $\mathrm{n}$ features requires

$$
n2^{n-1}
$$

coalition evaluations.

Therefore, ignoring the cost of an individual model evaluation, the overall computational complexity can be approximated as

$$
O\left(hn2^{n-1}M\right).
$$

This exponential dependence on $\mathrm{n}$ is the fundamental scalability limitation of exact Shapley-value computation. While increasing the number of explained samples or background samples primarily introduces linear scaling, increasing the number of features causes the number of required coalition evaluations to grow exponentially.


In [11]:
X_background = X_train.sample(n=n_background) 
x = X_test.iloc[0:n_samples*10]

start_t = perf_counter()
shap_explainer = shap.ExactExplainer(model.predict, X_background)
shap_values = shap_explainer(x)
print(f"{'SHAP Library':<30}: {perf_counter() - start_t:.3f}s")

SHAP Library                  : 0.065s


In [12]:
X_background = X_train.sample(n=n_background*5) 
x = X_test.iloc[0:n_samples*10]

start_t = perf_counter()
optimized_exact_explainer = ExactExplainerOptimized(model, X_background)
optimized_exact_shap_values = optimized_exact_explainer.explain(x)
print(f"{'Optimized Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

Optimized Exact Explainer     : 4.838s


# Dataset with more features

The Ames Housing dataset contains 2,930 residential property sales and 82 columns, including numerous property characteristics that make it well suited for demonstrating the computational limitations of exact SHAP value computation.


In [13]:
ames_housing = pd.read_csv("../data/ames_housing_dataset.csv")
ames_housing.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### 'Cleaning' the dataset

As we are not concerned with model accuracy or other performance metrics, we drop all non-numeric columns to allow the LinearRegression model to work with the data.


In [14]:
numeric_features = [col for col, dtype in zip(ames_housing.columns, ames_housing.dtypes) if dtype in ("int64", "float64")]
target = "SalePrice"

# Limitations of Exact Explainer

The code below demonstrates the computational limitations of calculating every possible feature combination and, consequently, the marginal contribution of all features. Even though our test instance `x` and background data `X_background` are relatively small, the time required for the exact calculation grows exponentially with the number of features.

The Exact Explainer evaluates $2^n$ possible feature masks, where $n$ is the number of features. This means that adding just one additional feature doubles the number of required evaluations. In this example, we calculate 

$2^5 = 32$ (~0.93s), \
$2^{10} = 1,024$ (~0.03s), \
$2^{15} = 32,768$ (~1.28s), \
$2^{20} = 1,048,576$ (~101.72s) and \
$2^{21} = 2,097,152$ (~567.39s) 

masked evaluations.

Hence, the number of evaluations quickly becomes impractical as the number of features increases. By default, SHAP's ExactExplainer is limited to 100,000 evaluations. We can calculate the corresponding number of features:

$$
2^n = 100,000
$$

$$
n = \lfloor \log_2(100,000) \rfloor = 16
$$


In [15]:
prev_t = None
for n_features in (5, 10, 15, 20, 21):
    X = ames_housing[numeric_features[:n_features]]
    y = ames_housing[[target]]

    X = X.dropna()
    y = y.loc[X.index]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    X_background = X_train.sample(n=100, random_state=42)
    x = X_test[0:3]

    start_t = perf_counter()
    explainer = shap.ExactExplainer(model.predict, X_background)
    shap_values = explainer(x, max_evals=2_097_152, silent=True)
    end_t = perf_counter() - start_t
    print(f"{n_features} features: {end_t:.3f}s - {end_t / prev_t if prev_t else 0:.2f}x slower")
    prev_t = end_t

5 features: 0.938s - 0.00x slower
10 features: 0.030s - 0.03x slower
15 features: 1.280s - 43.02x slower
20 features: 101.723s - 79.44x slower
21 features: 567.396s - 5.58x slower
